# SenticNet API Comparison

> Part of: *BERT vs LLM vs SenticNet: A Multi-Domain Sentiment Comparison*

SenticNet takes a fundamentally different approach from both BERT and the LLM. It is a knowledge-based commonsense reasoning system — not a statistical model trained on text corpora. This makes it interpretable in a way the other two are not.

**APIs used (ensemble key runs all in one call):**

| Signal | Field | What it gives |
|--------|-------|---------------|
| Polarity | `polarity` | POSITIVE / NEGATIVE / NEUTRAL |
| Intensity | `intensity` | Numeric sentiment strength |
| Emotions | `emotions` | JOY, SADNESS, ECSTASY, ANGER, etc. |
| Aspects | `aspects` | Named aspect phrases |
| Sarcasm | `sarcasm` | Sarcasm detection string |

**My run:** 200 samples per domain (600 total). API latency makes full-dataset runs impractical.

---

## Setup

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, '../src')
from data_utils import load_all_domains, SEED, DOMAINS
from sentic_utils import run_sentic_inference, summarize_sentic_results, get_emotion_distribution

warnings.filterwarnings('ignore')
load_dotenv(dotenv_path='../.env')

KEYS = {
    'ensemble':    os.getenv('SENTIC_ENSEMBLE_KEY',    ''),
    'polarity':    os.getenv('SENTIC_POLARITY_KEY',    ''),
    'emotion':     os.getenv('SENTIC_EMOTION_KEY',     ''),
    'sarcasm':     os.getenv('SENTIC_SARCASM_KEY',     ''),
    'subjectivity': os.getenv('SENTIC_SUBJECTIVITY_KEY', ''),
    'toxicity':    os.getenv('SENTIC_TOXICITY_KEY',    ''),
}

Path('../results').mkdir(exist_ok=True)
Path('../plots').mkdir(exist_ok=True)

print('Keys loaded:')
for name, key in KEYS.items():
    print(f'  {name}: {key[:6]}...' if key else f'  {name}: NOT SET')

## Load Datasets

In [ ]:
datasets = load_all_domains(n_per_domain=2000, dataset_dir='../datasets')

EXPLORE_N = 200
print(f'Running Sentic on {EXPLORE_N} samples per domain.')
print('Set EXPLORE_N = 2000 for the full dataset (expect ~20 min per domain).')

## Quick API Test

Verifying the API is responding correctly with a few test cases.

In [ ]:
from sentic_utils import call_sentic_api, clean_for_sentic

test_texts = [
    'This movie was absolutely fantastic. I loved every minute of it.',
    'Terrible waste of time. I want my money back.',
    'Yeah right, like this film could ever be considered good. Laughably bad.',  # sarcasm
    'The acting was great but the plot was confusing and slow.',  # mixed
]

print('=== API TEST CALLS ===')
for text in test_texts:
    result, latency = call_sentic_api(text, KEYS['ensemble'])
    print(f'Input:    {text[:60]}...' if len(text) > 60 else f'Input:    {text}')
    print(f'Polarity: {result["polarity"]} | Intensity: {result["intensity"]} | Emotion: {result["emotions"]}')
    print(f'Sarcasm:  {result["sarcasm"]} | Is_sarcastic: {result["is_sarcastic"]}')
    print(f'Aspects:  {result["aspects"]}')
    print(f'Latency:  {latency*1000:.0f}ms')
    print()

### What the API Reveals

From my test calls and full inference run, I observe that SenticNet provides qualitatively different information from BERT or the LLM:

- **Emotion labels** (ECSTASY, JOY, ANXIETY, GRIEF, ANGER) capture the *character* of sentiment, not just its direction — a dimension unavailable from binary classifiers.
- **Intensity** provides a gradient signal rather than a binary classification.
- **Sarcasm detection** is a dedicated parser that I cross-reference with BERT failure rates in the cross-domain analysis.

## Batch Inference — Ensemble API

⚠️ This is slow (~2,000 ms/sample average in my run).
- 200 samples ≈ 7–10 min per domain (actual: IMDb 2618ms avg, Twitter 1626ms avg, Amazon 1986ms avg)

Results are cached — run once, load from CSV after.

In [ ]:
sentic_results = {}
summaries = []

for domain, df in datasets.items():
    cache_path = f'../results/sentic_{domain}.csv'

    if Path(cache_path).exists():
        print(f'Loading cached {domain} results from {cache_path}')
        sentic_df = pd.read_csv(cache_path)
        sentic_results[domain] = sentic_df
        summary = summarize_sentic_results(sentic_df, df.iloc[:len(sentic_df)]['label'], domain=domain)
        summaries.append(summary)
        print()
        continue

    subset = df.head(EXPLORE_N)

    print(f'--- {domain.upper()} ({EXPLORE_N} samples) ---')
    sentic_df = run_sentic_inference(
        subset['text_clean'].tolist(),
        key=KEYS['ensemble'],
        api_name=f'ensemble/{domain}',
        sleep_between=0.15
    )
    sentic_df['ground_truth'] = subset['label'].values
    sentic_df['text'] = subset['text_clean'].values
    sentic_df['word_count'] = subset['word_count'].values
    sentic_df['domain'] = domain

    sentic_df.to_csv(cache_path, index=False)
    sentic_results[domain] = sentic_df

    summary = summarize_sentic_results(sentic_df, subset['label'], domain=domain)
    summaries.append(summary)
    print()

## Summary Table

In [ ]:
if summaries:
    summary_df = pd.DataFrame(summaries)
    summary_df['accuracy'] = summary_df['accuracy'].map('{:.1%}'.format)
    summary_df['neutral_rate'] = summary_df['neutral_rate'].map('{:.1%}'.format)
    summary_df['avg_latency_ms'] = summary_df['avg_latency_ms'].map('{:.0f} ms'.format)
    summary_df['sarcasm_rate'] = summary_df['sarcasm_rate'].map('{:.1%}'.format)
    display(summary_df)

### On Neutral Predictions — Empirical Findings

Contrary to my theoretical expectation that SenticNet would exhibit meaningful neutral abstention, the empirical results show near-zero neutral rates:

| Domain | Accuracy | Neutral Rate | Latency |
|--------|----------|--------------|----------|
| **IMDb** | 64.5% | **0.0%** | 2,618 ms |
| **Twitter** | 65.1% | **2.5%** | 1,626 ms |
| **Amazon** | 71.4% | **0.5%** | 1,986 ms |

SenticNet almost always produces a binary prediction rather than abstaining. This means the low overall accuracy (64–71%) cannot be explained by abstention on hard cases — the system is actively misclassifying a substantial fraction. This was the most surprising finding of the SenticNet evaluation: rather than exhibiting selective coverage with high precision, it covers nearly everything with moderate accuracy.

## Emotion Distribution by Domain

This analysis is unique to SenticNet — BERT and the LLM produce no emotion-category output.

In [ ]:
fig, axes = plt.subplots(1, len(sentic_results), figsize=(14, 5))
if len(sentic_results) == 1:
    axes = [axes]

for ax, (domain, df) in zip(axes, sentic_results.items()):
    emo_dist = get_emotion_distribution(df).head(10)
    emo_dist.plot(kind='barh', ax=ax, color='steelblue', alpha=0.8)
    ax.set_title(f'{domain.upper()}\nEmotion Distribution (top 10)')
    ax.set_xlabel('Count')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../plots/sentic_emotion_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to plots/sentic_emotion_distribution.png')

### Emotion Distribution — Findings

The emotion label distributions from my 200-sample-per-domain run reveal qualitative differences in the emotional profiles of each domain:

- **IMDb** shows the widest emotional range, with emotion combinations including ANGER/ANXIETY, ECSTASY/ENTHUSIASM, and TERROR/GRIEF appearing across different samples. This is consistent with the expressive, narrative-driven register of film reviews, which encode a broader affective vocabulary than functional product writing.

- **Twitter** shows concentration at the extremes: JOY (100%) and ECSTASY (100%) dominate the positive-polarity samples, while "NO EMOTIONS DETECTED" is the most frequent single category (n=5, 2.5% of samples). This reflects Twitter's short text length — insufficient context for fine-grained emotion labeling. The high-frequency pure-JOY labels for positive tweets suggest the API treats enthusiastic short text as maximally positive.

- **Amazon** clusters around ECSTASY, ENTHUSIASM, JOY, and BLISS on positive reviews — reflecting a more functional, transactional register where positive sentiment tends to be expressed as straightforward satisfaction rather than complex narrative affect.

These domain-level differences in emotional profile are a novel contribution of this analysis — they are invisible to binary accuracy metrics and cannot be produced by BERT or the LLM as configured in this study.

## Sarcasm Analysis

I examine how frequently SenticNet detects sarcasm and whether its detections correlate with BERT failure rates.

In [ ]:
for domain, df in sentic_results.items():
    sarcastic = df[df['is_sarcastic'] == True]
    print(f'{domain.upper()}: {len(sarcastic)} sarcastic samples detected ({len(sarcastic)/len(df):.1%})')

    if len(sarcastic) > 0:
        print('  Sample sarcastic texts:')
        for _, row in sarcastic.head(3).iterrows():
            print(f'  [{"POS" if row["ground_truth"] == 1 else "NEG"}] {str(row["text"])[:200]}...')
            print(f'  Sarcasm signal: {row["sarcasm"]}')
        print()
    print()

## Sarcasm + BERT Accuracy Crosscheck

In [ ]:
for domain in DOMAINS:
    if domain not in sentic_results:
        continue

    sentic_df = sentic_results[domain]
    sarcastic_mask = sentic_df['is_sarcastic'].fillna(False)
    n_sarcastic = sarcastic_mask.sum()

    if n_sarcastic == 0:
        print(f'{domain.upper()}: no sarcastic samples detected, skipping.')
        continue

    bert_path = f'../results/bert_{domain}.csv'
    if Path(bert_path).exists():
        bert_df = pd.read_csv(bert_path).head(len(sentic_df))
        sarcastic_idx = sentic_df[sarcastic_mask].index
        bert_sarcastic_acc = (bert_df.loc[sarcastic_idx, 'bert_pred'] ==
                              sentic_df.loc[sarcastic_idx, 'ground_truth']).mean()
        print(f'{domain.upper()}: BERT accuracy on sarcastic samples: {bert_sarcastic_acc:.1%} '
              f'(vs {bert_df["correct"].mean():.1%} overall)')
    else:
        print(f'{domain.upper()}: BERT results not found for comparison')

    print()

### Sarcasm Detection — Unexpected Finding

The empirical sarcasm analysis produced a counterintuitive result:

| Domain | Sarcastic (n) | Rate | BERT on Sarcastic | BERT Overall | Δ |
|--------|--------------|------|-------------------|--------------|----|
| **IMDb** | 9 | 4.5% | **100.0%** | 89.5% | +10.5% |
| **Twitter** | 13 | 6.5% | **100.0%** | 80.0% | +20.0% |
| **Amazon** | 10 | 5.0% | **90.0%** | 90.0% | 0.0% |

Contrary to my hypothesis that sarcasm-flagged samples would correspond to lower BERT accuracy, I found the opposite: BERT accuracy on sarcasm-flagged samples is equal to or *higher* than its overall accuracy in all three domains. This suggests that either (a) SenticNet's sarcasm detector identifies a distinctive type of ironic text that is structurally easier for BERT to classify correctly, or (b) the sarcasm detection rate is low enough (4.5–6.5%) that sampling noise dominates the comparison. At 9–13 sarcastic samples per domain, I cannot draw statistically reliable conclusions about sarcasm-specific accuracy. This is a limitation of the 200-sample subset used for SenticNet evaluation.

## Aspect Extraction

In [ ]:
for domain, df in sentic_results.items():
    has_aspects = df[~df['aspects'].str.contains('No aspects', na=True, case=False)]
    print(f'{domain.upper()}: {len(has_aspects)}/{len(df)} samples had aspects detected')

    if len(has_aspects) > 0:
        sample_aspects = has_aspects.sample(min(5, len(has_aspects)), random_state=42)
        for _, row in sample_aspects.iterrows():
            print(f'  [{"POS" if row["ground_truth"] == 1 else "NEG"}] aspects: {row["aspects"]}')
        print()
    print()

## Latency Comparison

In [ ]:
latency_summary = []
for domain, df in sentic_results.items():
    latency_summary.append({
        'domain': domain,
        'method': 'SenticNet',
        'avg_latency_ms': df['sentic_latency_s'].mean() * 1000,
        'p95_latency_ms': df['sentic_latency_s'].quantile(0.95) * 1000,
    })

latency_df = pd.DataFrame(latency_summary)
print('SenticNet latency summary:')
display(latency_df)

print()
print('For comparison (from my completed runs):')
print('  BERT (CPU):  IMDb=107ms | Twitter=27ms | Amazon=47ms')
print('  LLM (API):   IMDb=1109ms | Twitter=795ms | Amazon=1066ms')
print('  SenticNet:   see above')

## Conclusions

Based on running SenticNet's ensemble API on 600 samples (200 per domain), I draw the following conclusions:

1. **SenticNet's binary classification accuracy (64.5–71.4%) is substantially lower than both BERT and the LLM.** This is the central empirical finding of this notebook. Amazon was SenticNet's strongest domain (71.4%), followed by Twitter (65.1%) and IMDb (64.5%). This ordering is the inverse of BERT's domain ranking, suggesting that SenticNet's commonsense knowledge graph is better calibrated for functional product-review language than for the structured narrative discourse of film reviews.

2. **The neutral abstention mechanism does not function as expected.** I predicted that SenticNet would abstain on ambiguous samples — providing selective high-precision coverage. Instead, I found near-zero neutral rates (0–2.5%), meaning the system produces binary predictions on nearly every input regardless of confidence. This removes a key theoretical advantage of the knowledge-based approach.

3. **Latency is prohibitively high for production use.** At 1,626–2,618 ms per sample average (compared to 27–108 ms for BERT and 795–1,109 ms for the LLM), SenticNet cannot serve as a primary classifier in any latency-sensitive pipeline. Its role must be limited to offline analysis or selective escalation on small subsets.

4. **Emotion distributions differ meaningfully across domains**, with IMDb showing the broadest emotional vocabulary, Twitter showing high-intensity single-emotion labels (JOY 100%), and Amazon clustering around contentment-adjacent emotions (ECSTASY, ENTHUSIASM, BLISS). This domain-specific emotional profiling is a genuine analytical capability that binary classifiers cannot replicate.

5. **The sarcasm detection finding is inconclusive at 200 samples per domain.** Sarcasm rates of 4.5–6.5% yield sample sizes of 9–13 per domain — too small for reliable cross-method comparison. This is a limitation that would require a larger SenticNet evaluation run to resolve.

---

*Results saved to `results/sentic_{domain}.csv` for use in `cross_domain_analysis.ipynb`.*